# Plots

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/Apoptosis_EXP58_62_63/output"
filter_dapi = TRUE

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/EXP58/output/setB_T2/4_plots/EXP58_setB_T2_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP58/output/setB_T3/4_plots/EXP58_setB_T3_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP62/output/setB_T2/4_plots/EXP62_setB_T2_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP62/output/setB_T3/4_plots/EXP62_setB_T3_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP63/output/setB_T3/4_plots/EXP63_setB_T3_analysis_summary.csv"
)

In [ ]:
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F63", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP = c("TRUE"= "#86AB30","FALSE"="#8d8d8dff")

col_condition_3 = c("Developed" = "#285F62", 
               "Poor Quality" = "#CA4F33")


## 1. Extract summary files

In [ ]:
# Read and combine all files into one dataframe
merged_df <- analysis_summary_files %>%
  map_dfr(read_csv)

In [ ]:
tbl <- merged_df %>%
  group_by( sample_name) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") 
tbl

In [ ]:
tbl <- merged_df %>%
  group_by(exp, exp_sub) %>%
  summarise(n = n_distinct(image), .groups = "drop") 
tbl

## 2. Preprocess

In [ ]:
colnames(merged_df)

# Filter cells with higher dapi levels
if (filter_dapi) {
  merged_df <- merged_df %>% 
    filter(Mean_dapi < DAPI_thresh)
}

In [ ]:
unique(merged_df$sample_name)

In [ ]:

order_sample <- c(
  "D4_GR",  "D4_GrevRrev", "D4_GrevR", "D4_RrevG",
  "D6_GR_D", "D6_GR_F", "D6_GrevRrev_D", "D6_GrevRrev_F", 
  "D6_GrevR_D", "D6_GrevR_F", "D6_RrevG_D", "D6_RrevG_F")
merged_df <- merged_df %>%
  mutate(sample_name = factor(sample_name, levels = order_sample))

## Plot 

### E) %pos marker

In [ ]:
plot_pct_bar_points <- function(
  data,                               # e.g., summary_df
  pct = pct_caspase3,                    # <-- column with % values to plot
  sample = sample_name,               # sample/category column
  condition = condition,              # grouping/fill column
  out_dir,                    # folder to save (optional)
  title = NULL,                       # default built from pct col name if NULL
  palette = NULL,                     # named vector for fill
  w = 3, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  point_size = 0.3,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct      <- enquo(pct)
  sample   <- enquo(sample)
  condition<- enquo(condition)

  # default title from pct column name if not supplied
  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # per-sample means (by condition) for the chosen pct column
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(mean_pct = mean(!!pct, na.rm = TRUE), .groups = "drop")

  # build plot (reverse sample order, flip coords)
  p <- ggplot(data, aes(x = fct_rev(!!sample), y = !!pct)) +
    geom_col(
      data = means_df,
      aes(y = mean_pct, fill = !!condition),
      width = bar_width
    ) +
    geom_point(
      size = point_size, alpha = point_alpha,
      position = position_jitter(width = jitter_width),
      na.rm = TRUE
    ) +
    labs(x = "", y = "% positive", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
    coord_flip()+
    theme(legend.position = "none")

  if (!is.null(palette)) {
    p <- p + scale_fill_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("%s.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
summary_df <- merged_df %>%
  group_by(exp, image, sample_name, condition) %>%
  summarise(
    n = n(),
    
    # Calculate simple percentages based on your existing boolean columns
    pct_caspase3 = 100 * mean(caspase3pos, na.rm = TRUE),
    #pct_NANOG = 100 * mean(NANOGpos, na.rm = TRUE),
    pct_GFP   = 100 * mean(GFPpos,   na.rm = TRUE),
    
    # Calculate negative cells (None of the markers are positive)
    #pct_negative = 100 * mean(!(GFPpos | NANOGpos | caspase3pos), na.rm = TRUE),
    
    # Calculate double positives (Both markers are positive)
    #pct_double_caspase3_NANOG = 100 * mean(caspase3pos & NANOGpos, na.rm = TRUE),
    
    .groups = "drop"
  )

# Plot % caspase3+
plot_pct_bar_points(summary_df, pct = pct_caspase3, palette = col_condition, out_dir = out_dir, 
                    title = "E_pctcaspase3+")
# Plot % GFP+
plot_pct_bar_points(summary_df, pct = pct_GFP, palette = col_condition,out_dir = out_dir,
                    title = "E_pctGFP+")




#

In [ ]:
head(summary_df)

In [ ]:
df2 <- summary_df %>%
  separate(
    sample_name,
    into = c("days", "condition2", "state"),
    sep = "_",
    remove = FALSE
  ) %>%
  mutate(
    condition = case_when(
      condition == 'G_R' ~ "control",
      condition == 'Grev_Rrev' ~ "reversine",
      TRUE ~ "mosaic"
    )
  )

In [ ]:
unique(summary_df$condition)

In [ ]:
unique(df2$state)

In [ ]:
tail(df2)

In [ ]:
title = "per_caspase3+"
w <- 2.7
h <- 1.7
options(repr.plot.width=w, repr.plot.height=h)


df2_sub <- df2 %>%
  filter(state == "D" | is.na(state))

p = ggplot(df2_sub, aes(x = condition, y =pct_caspase3)) +  # dots for each file
    stat_summary( aes(fill = condition), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6, fill = "grey") +   # error bars
    geom_jitter(
      aes(fill = condition),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = "#B165A0"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%caspase3+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 52), expand = c(0, 0)) +
      facet_wrap(~days)


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
library(emmeans)
library(dplyr)
library(purrr)
library(rstatix)

anova_res <- df2_sub %>%
  group_by(days) %>%
  anova_test(pct_caspase3 ~ condition) %>%
  add_significance(p.col = "p")

anova_res

dunnett_res <- df2_sub %>%
  group_by(days) %>%
  group_modify(~ {
    fit <- aov(pct_caspase3 ~ condition, data = .x)

    contrast(
      emmeans(fit, ~ condition),
      method = "trt.vs.ctrl",
      ref = "control"
    ) %>%
      as.data.frame()
  }) %>%
  ungroup() %>%
  mutate(
    p.signif = case_when(
      p.value <= 0.0001 ~ "****",
      p.value <= 0.001  ~ "***",
      p.value <= 0.01   ~ "**",
      p.value <= 0.05   ~ "*",
      TRUE              ~ "ns"
    )
  )

dunnett_res

In [ ]:
library(dplyr); library(tidyr); library(purrr); library(car)

check_test <- function(data, group_var = "condition", value_var = "pct_caspase3",
                       conditions = c("control", "mosaic", "reversine"), alpha = 0.05) {

  d <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    mutate(across(all_of(group_var), ~ factor(.x, levels = conditions))) %>%
    droplevels()

  d %>%
    group_by(days) %>%
    group_modify(~ {
      g <- split(.x[[value_var]], .x[[group_var]])[conditions]
      k <- length(g)
      ns <- sapply(g, function(x) sum(!is.na(x)))

      if (!all(ns >= 3)) {
        return(tibble(
          k = k, n = paste(ns, collapse = "/"),
          shapiro_p_min = NA_real_, levene_p = NA_real_, normal = NA,
          recommended = "too few points (use non-parametric / be cautious)"
        ))
      }

      # normality: test every group, take the worst
      sps <- sapply(g, function(x) shapiro.test(x)$p.value)
      normal <- all(sps > alpha)

      # homogeneity of variance across all k groups (Levene, robust to non-normality)
      lev_p <- tryCatch(
        car::leveneTest(.x[[value_var]] ~ .x[[group_var]])[1, "Pr(>F)"],
        error = function(e) NA_real_
      )
      equal_var <- !is.na(lev_p) && lev_p > alpha

      rec <- if (k == 2) {
        if (normal) { if (equal_var) "Student t-test" else "Welch t-test" }
        else "Wilcoxon rank-sum test"
      } else {
        if (normal) { if (equal_var) "One-way ANOVA + Tukey HSD"
                      else "Welch's ANOVA + Games-Howell" }
        else "Kruskal-Wallis + Dunn's test"
      }

      tibble(
        k = k, n = paste(ns, collapse = "/"),
        shapiro_p_min = min(sps), levene_p = lev_p, normal = normal,
        recommended = rec
      )
    }) %>%
    ungroup()
}

check_test(df2_sub)

In [ ]:
kw_res <- df2_sub %>%
  group_by(days) %>%
  kruskal_test(pct_caspase3 ~ condition) %>%
  mutate(
    stars = case_when(
      p < 0.0001 ~ "****",
      p < 0.001  ~ "***",
      p < 0.01   ~ "**",
      p < 0.05   ~ "*",
      TRUE       ~ "ns"
    )
  )
kw_res

## Caspase withinGFP+

In [ ]:
GFPpos <- merged_df %>% filter(GFPpos == TRUE)

summary_df_sub <- GFPpos %>%
  group_by(image, sample_name, condition, timepoint, condition_2) %>%
  summarise(
    n = n(),
    #pct_GATA3 = 100 * mean(GATA3_norm > thr$GATA3_norm, na.rm = TRUE),
    pct_caspase3 = 100 * mean(caspase3pos, na.rm = TRUE),
    pct_GFP = 100 * mean(GFPpos, na.rm = TRUE),
    .groups = "drop"
  )

head(summary_df_sub)

In [ ]:
# Plot %caspase3+
plot_pct_bar_points(summary_df_sub, pct = pct_caspase3, palette = col_condition,out_dir = out_dir,
                    title = "E_pctcaspase3+_GFP+",
  y_max = 50)

In [ ]:
summary_df_sub_sub = summary_df_sub %>% filter(timepoint == "D4")


In [ ]:
unique(summary_df_sub_sub$condition)

In [ ]:

order_sample <- c(
'G_R',  'Rrev_G','Grev_Rrev',  'Grev_R')
summary_df_sub_sub <- summary_df_sub_sub%>%
  mutate(condition = factor(condition, levels = order_sample))

In [ ]:
title = "proportion developed"
w <- 3
h <- 1.6
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(summary_df_sub_sub, aes(x = condition , y = pct_caspase3, fill  = condition_2)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75), alpha = 0.6, width = 0.6, fill = "#aaaeaa") +   # error bars
    geom_jitter(
      aes(color = condition_2),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.8, alpha = 0.9
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "#080908")+
    labs(
      title = title,
      y = "proportion caspase3+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 0, hjust = 0),
      #legend.position = "none", 
      legend.key.size = unit(0.3, "cm")
      
    ) +
      scale_y_continuous(limits = c(0, 52), expand = c(0, 0))+
      #facet_wrap( ~ condition_2) +
      scale_color_manual(values=col_condition_2)+
    coord_flip()

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

ps

In [ ]:
# Wilcoxon Test (Non-parametric, safer for small N or percentages)
# test_result <- wilcox.test(pct_GATA3 ~ condition, data = summary_df)
# print(test_result)

# T-test (Parametric, use if data is normally distributed)
t_test_result <- t.test( pct_caspase3~ condition, data = summary_df_sub_sub %>% filter(sample_name %in% c("Grev_R", "G_R")))
print(t_test_result)

In [ ]:
summary_df_sub_sub %>%
  #group_by(condition) %>%
  pairwise_wilcox_test(
    pct_caspase3~ condition,
    p.adjust.method = "none"
  ) 

## Save

In [ ]:
head(merged_df)

In [ ]:
write.csv(merged_df, file.path(out_dir, sprintf("%s_analysis_summary.csv", "merged")), row.names = FALSE)